# Companio: audio distress (scream) detection prototype

**Goal.** Given a short audio clip, estimate the probability that it contains a human scream. The backend would use this score to decide whether to start an SOS (high), ask the user to confirm (medium), or ignore it (low).

**Approach.** Transfer learning. A pretrained audio model (YAMNet, trained on Google's AudioSet) turns each clip into an embedding. A small classifier is trained on those embeddings. This needs far less data and compute than training a neural network from scratch.

**Status: prototype.** This notebook is not yet connected to the Spring Boot backend. The numbers it produces come from public datasets, not real emergencies, and the report should say so.

**What it produces (for the report and slides)**
1. A baseline with no training (YAMNet's built-in "Screaming" scores)
2. A trained classifier compared against that baseline
3. Precision, recall, F1, ROC-AUC, PR-AUC and a confusion matrix on a held-out test set
4. HIGH / MEDIUM / LOW thresholds chosen from validation data
5. A false-positive analysis: which everyday sounds fool the model

## How to run

1. Open this file in Google Colab (colab.research.google.com, then File, Upload notebook).
2. Set the runtime to GPU (Runtime, Change runtime type, T4 GPU). It works on CPU too, only slower.
3. Download the **Human Screaming Detection Dataset** zip from Kaggle (linked in the README of the GitHub repo `whats2000/HumanScreamingDetection`). Check its license and cite it in your report.
4. Run the cells from top to bottom. In section 3 you will load that zip (the upload cell for small zips, or the Google Drive cell for big ones), read the folder listing it prints, and set `SCREAM_DIR` in section 4.

## 1. Setup

In [ ]:
# ---- 1. setup ----
!pip install -q librosa soundfile
import os, zipfile, random, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf, tensorflow_hub as hub
import librosa, joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay, roc_auc_score,
                             average_precision_score, precision_recall_curve,
                             f1_score, precision_score, recall_score)
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED); random.seed(SEED)
SR = 16000            # YAMNet expects 16 kHz mono audio
MAX_SECONDS = 10      # only the first 10 s of each clip are used
AUDIO_EXT = ('.wav', '.mp3', '.ogg', '.flac', '.m4a')
print('TensorFlow', tf.__version__)

## 2. Load YAMNet

YAMNet is a pretrained audio-event model (521 sound classes from AudioSet). Two things are used from it: its 1024-number **embedding** for each ~1 second of audio (the features for our classifier), and its own class scores, which include "Screaming" (used for the no-training baseline).

In [ ]:
# ---- 2. yamnet ----
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')
class_map = pd.read_csv(yamnet.class_map_path().numpy().decode('utf-8'))
class_names = class_map['display_name'].tolist()
SCREAM_CLASSES = ['Screaming', 'Shout', 'Yell', 'Children shouting']
scream_idx = [class_names.index(c) for c in SCREAM_CLASSES if c in class_names]
print(len(class_names), 'classes. Scream-related ones used for the baseline:', [class_names[i] for i in scream_idx])

## 3. Data

* **Negatives (not scream):** ESC-50, 2000 five-second clips across 50 everyday sound categories (dog, siren, crying baby, laughing, door knock, rain ...). It comes with 5 predefined folds so clips from the same source never sit in both train and test.
* **Positives (scream):** the Kaggle Human Screaming Detection Dataset you download yourself.

In [ ]:
# ---- 3a. esc-50 negatives ----
!wget -q https://github.com/karolpiczak/ESC-50/archive/master.zip -O esc50.zip
!unzip -q -o esc50.zip
ESC_DIR = 'ESC-50-master'
esc = pd.read_csv(f'{ESC_DIR}/meta/esc50.csv')
print(esc.shape, 'clips,', esc['category'].nunique(), 'categories')

In [ ]:
# ---- 3b. kaggle scream dataset: upload the zip you downloaded ----
# (For a big zip, mount Google Drive instead: from google.colab import drive; drive.mount('/content/drive'))
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(name) as z:
            z.extractall('kaggle_scream')
print('Folders that contain audio:')
for root, dirs, fs in os.walk('kaggle_scream'):
    n = sum(f.lower().endswith(AUDIO_EXT) for f in fs)
    if n:
        print(f'  {root}  ->  {n} audio files')

### 3b (alternative). Big zip from Google Drive

Use this cell **instead of** the upload cell above if the zip is large (the Kaggle dataset is about 4 GB). Put the zip in your Google Drive first, set `ZIP_PATH` to its file name, then run the cell. It unzips onto Colab's local disk, which is much faster than reading from Drive.

In [ ]:
# ---- 3b-alt. kaggle zip from Google Drive ----
from google.colab import drive
drive.mount('/content/drive')
ZIP_PATH = '/content/drive/MyDrive/PUT_YOUR_ZIP_FILE_NAME_HERE.zip'
assert os.path.exists(ZIP_PATH), 'ZIP_PATH not found. Check the file name in the Files panel (left side, Drive folder).'
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('kaggle_scream')
print('Folders that contain audio:')
for root, dirs, fs in os.walk('kaggle_scream'):
    n = sum(f.lower().endswith(AUDIO_EXT) for f in fs)
    if n:
        print(f'  {root}  ->  {n} audio files')

## 4. Choose the folders

Read the listing above. Set `SCREAM_DIR` to the folder holding the **scream** clips. If the dataset also has a folder of **non-scream** clips, set `NONSCREAM_DIR` too (otherwise leave it `None` and only ESC-50 provides negatives).

In [ ]:
# ---- 4. paths: EDIT THESE ----
SCREAM_DIR = 'kaggle_scream/PUT_THE_SCREAM_FOLDER_HERE'
NONSCREAM_DIR = None   # e.g. 'kaggle_scream/PUT_THE_NON_SCREAM_FOLDER_HERE'

assert os.path.isdir(SCREAM_DIR), 'SCREAM_DIR does not exist. Copy a folder path from the listing in section 3b.'

MAX_PER_CLASS = 1500   # cap per Kaggle class so feature extraction stays around 20 minutes

def list_audio(folder):
    out = []
    for root, _, fs in os.walk(folder):
        out += [os.path.join(root, f) for f in fs if f.lower().endswith(AUDIO_EXT)]
    return sorted(out)

def cap(paths, name):
    print(f'{name}: {len(paths)} clips found', end='')
    if len(paths) > MAX_PER_CLASS:
        idx = np.random.default_rng(SEED).choice(len(paths), MAX_PER_CLASS, replace=False)
        paths = [paths[i] for i in sorted(idx)]
    print(f', using {len(paths)}')
    return paths

rng = np.random.default_rng(SEED)
def random_split(n):
    # 60/20/20 train/val/test, matching ESC-50's 3/1/1 folds
    r = rng.permutation(n); s = np.empty(n, dtype=object)
    s[r[:int(.6 * n)]] = 'train'
    s[r[int(.6 * n):int(.8 * n)]] = 'val'
    s[r[int(.8 * n):]] = 'test'
    return s

rows = []
pos = cap(list_audio(SCREAM_DIR), 'scream')
for p, s in zip(pos, random_split(len(pos))):
    rows.append(dict(path=p, label=1, category='scream', split=s))
for _, r in esc.iterrows():
    split = 'train' if r['fold'] <= 3 else 'val' if r['fold'] == 4 else 'test'
    rows.append(dict(path=f"{ESC_DIR}/audio/{r['filename']}", label=0, category=r['category'], split=split))
if NONSCREAM_DIR:
    neg = cap(list_audio(NONSCREAM_DIR), 'non-scream')
    for p, s in zip(neg, random_split(len(neg))):
        rows.append(dict(path=p, label=0, category='kaggle_nonscream', split=s))
meta = pd.DataFrame(rows)
print(meta.groupby(['split', 'label']).size().unstack(fill_value=0).rename(columns={0: 'not scream', 1: 'scream'}))

## 5. Extract features

Each clip becomes a 2048-number vector: the mean and the max of YAMNet's embeddings over time. The max helps when a short scream sits inside a longer, quieter clip. The first run takes several minutes; results are cached in `features.npz`, so re-running is instant.

In [ ]:
# ---- 5. feature extraction ----
def load_wave(path):
    wav, _ = librosa.load(path, sr=SR, mono=True, duration=MAX_SECONDS)
    if len(wav) < SR:                      # YAMNet needs at least ~1 s
        wav = np.pad(wav, (0, SR - len(wav)))
    return wav.astype(np.float32)

def extract(path):
    wav = load_wave(path)
    scores, emb, _ = yamnet(tf.constant(wav))
    scores, emb = scores.numpy(), emb.numpy()
    feat = np.concatenate([emb.mean(0), emb.max(0)])
    zero = float(scores[:, scream_idx].max()) if scream_idx else 0.0   # baseline score, no training
    return feat, zero

CACHE = 'features.npz'
if os.path.exists(CACHE) and os.path.exists('meta.csv'):
    d = np.load(CACHE); X, zero = d['X'], d['zero']
    meta = pd.read_csv('meta.csv')
else:
    feats, zs, ok = [], [], []
    for i, p in enumerate(meta['path']):
        try:
            f, z = extract(p); feats.append(f); zs.append(z); ok.append(True)
        except Exception as e:
            print('skipped', p, '-', e); ok.append(False)
        if i % 200 == 0:
            print(f'{i}/{len(meta)}')
    meta = meta[ok].reset_index(drop=True)
    X, zero = np.stack(feats), np.array(zs)
    np.savez(CACHE, X=X, zero=zero); meta.to_csv('meta.csv', index=False)
print('features:', X.shape)

## 6. Splits and a no-training baseline

In [ ]:
# ---- 6. splits and baseline ----
y = meta['label'].values
tr, va, te = [(meta['split'] == s).values for s in ('train', 'val', 'test')]

def best_f1_threshold(y_true, s):
    p, r, t = precision_recall_curve(y_true, s)
    f1 = 2 * p * r / np.maximum(p + r, 1e-9)
    return t[np.argmax(f1[:-1])]

def summarize(name, y_true, score, thr):
    pred = (score >= thr).astype(int)
    return dict(model=name,
                precision=precision_score(y_true, pred, zero_division=0),
                recall=recall_score(y_true, pred, zero_division=0),
                f1=f1_score(y_true, pred, zero_division=0),
                roc_auc=roc_auc_score(y_true, score),
                pr_auc=average_precision_score(y_true, score))

t0 = best_f1_threshold(y[va], zero[va])   # threshold picked on validation data only
base = summarize('Baseline: YAMNet built-in scream scores (no training)', y[te], zero[te], t0)
print(f'Baseline threshold (chosen on validation): {t0:.3f}')
pd.DataFrame([base]).round(3)

## 7. Train the classifier

Logistic regression on standardised embeddings, with class weights because scream clips and non-scream clips are unlikely to be equally numerous. The test set is used once, at the end of this section.

In [ ]:
# ---- 7. train and evaluate ----
clf = make_pipeline(StandardScaler(),
                    LogisticRegression(C=0.05, max_iter=3000, class_weight='balanced'))
clf.fit(X[tr], y[tr])
p_val = clf.predict_proba(X[va])[:, 1]
p_te = clf.predict_proba(X[te])[:, 1]

t_f1 = best_f1_threshold(y[va], p_val)
trained = summarize('Trained: YAMNet embeddings + logistic regression', y[te], p_te, t_f1)
print(f'Trained model threshold (max F1 on validation): {t_f1:.3f}\n')
print(classification_report(y[te], (p_te >= t_f1).astype(int), target_names=['not scream', 'scream'], digits=3))
ConfusionMatrixDisplay.from_predictions(y[te], (p_te >= t_f1).astype(int),
                                        display_labels=['not scream', 'scream'], cmap='Blues')
plt.title('Test-set confusion matrix'); plt.show()
pd.DataFrame([base, trained]).round(3)

## 8. HIGH / MEDIUM / LOW bands

In the app, a **HIGH** score would start an SOS straight away, a **MEDIUM** score would ask the user to confirm first, and a **LOW** score would be ignored. The cut-offs are chosen on the validation set:

* `t_high`: the lowest score at which precision is at least 90% (few false alarms)
* `t_med`: the highest score at which recall is still at least 90% (few missed screams)

A missed scream is worse than a false alarm, which is why the MEDIUM band exists: it catches screams the model is unsure about without alerting contacts every time.

In [ ]:
# ---- 8. thresholds and bands ----
def choose_thresholds(y_true, p, target_precision=0.90, target_recall=0.90):
    prec, rec, thr = precision_recall_curve(y_true, p)
    prec, rec = prec[:-1], rec[:-1]
    ok_p = np.where(prec >= target_precision)[0]
    ok_r = np.where(rec >= target_recall)[0]
    t_high = float(thr[ok_p].min()) if len(ok_p) else 0.90
    t_med = float(thr[ok_r].max()) if len(ok_r) else 0.30
    return min(t_med, t_high), t_high

t_med, t_high = choose_thresholds(y[va], p_val)
band = lambda p: 'HIGH' if p >= t_high else 'MEDIUM' if p >= t_med else 'LOW'
print(f't_med = {t_med:.3f}   t_high = {t_high:.3f}')
if t_high - t_med < 1e-6:
    print('Note: the two thresholds coincide on this data, so the MEDIUM band is empty. Lower target_precision or raise target_recall, or say so in the report.')

bands = pd.Series([band(p) for p in p_te], name='band')
truth = pd.Series(np.where(y[te] == 1, 'real scream', 'not scream'), name='true class')
table = pd.crosstab(truth, bands).reindex(columns=['HIGH', 'MEDIUM', 'LOW'], fill_value=0)
print('\nTest clips by band (counts):'); print(table)
print('\nShare of each true class falling in each band:'); print((table.div(table.sum(1), axis=0) * 100).round(1))

## 9. False-positive analysis

Which everyday sounds does the model wrongly flag? This uses the ESC-50 validation and test folds (clips the model never trained on), grouped by sound category. A model that screams "danger" at a laughing group or a crying baby will annoy users, so these are the cases worth discussing in the report.

In [ ]:
# ---- 9. false positives by category ----
vt = (va | te) & (y == 0) & (meta['category'] != 'kaggle_nonscream').values
p_vt = clf.predict_proba(X[vt])[:, 1]
fp = (pd.DataFrame({'category': meta.loc[vt, 'category'].values, 'flagged': p_vt >= t_med})
      .groupby('category')['flagged'].mean().sort_values(ascending=False) * 100)
print(fp.head(10).round(1).to_string())
fp.head(10).iloc[::-1].plot.barh(color='#D4235A')
plt.xlabel('% of clips flagged MEDIUM or HIGH'); plt.title('Everyday sounds most often mistaken for a scream')
plt.tight_layout(); plt.show()

## 10. Save the model and a prediction function

In [ ]:
# ---- 10. save and predict ----
joblib.dump({'model': clf, 't_med': t_med, 't_high': t_high, 'sr': SR, 'max_seconds': MAX_SECONDS},
            'distress_model.joblib')

def predict_clip(path):
    f, _ = extract(path)
    p = float(clf.predict_proba(f.reshape(1, -1))[0, 1])
    return {'distress_probability': round(p, 3), 'band': band(p)}

try:
    from google.colab import files
    files.download('distress_model.joblib')
except ImportError:
    pass
# example: predict_clip('kaggle_scream/.../some_clip.wav')

## Limitations to state in the report

* **Not real emergencies.** The clips are public recordings and film or game audio, so real-world performance will be lower. Phone microphones, distance, wind and background noise are not covered.
* **Binary and scream-only.** Crying, shouting for help and other distress sounds are future classes.
* **Small, random splits for the scream clips.** If several clips come from one source recording, the test score can be optimistic. ESC-50's folds protect the negatives but not necessarily the positives.
* **Server-side inference (planned).** The audio would be sent to a small Python service called by the Spring Boot backend. That service is not built yet.
* **Privacy.** Continuously analysing a person's audio is sensitive. A real deployment needs explicit consent and a plan for where audio is processed and stored.

## Related work to cite

* Scream detection with machine learning on balanced AudioSet data, validated on separate real-world audio (JMIR Formative Research, 2020; PMC7327591)
* YAMNet and AudioSet (Google)
* ESC-50 (Piczak, 2015)
* The Kaggle Human Screaming Detection Dataset (check its license and author before citing)

In [ ]:
%pip install -q kaggle


In [ ]:
import os
import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass.getpass(
    "Paste your Kaggle API token: "
)


Paste your Kaggle API token: ··········


In [ ]:
import os

DOWNLOAD_DIR = "/content/companio_data"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

!kaggle datasets download \
    -d whats2000/human-screaming-detection-dataset \
    -p /content/companio_data

Dataset URL: https://www.kaggle.com/datasets/whats2000/human-screaming-detection-dataset
License(s): MIT
100% 4.17G/4.17G [00:47<00:00, 94.9MB/s]



In [ ]:
import os
import shutil
import glob

DOWNLOAD_DIR = "/content/companio_data"

print("Downloaded files:")
for file in glob.glob(f"{DOWNLOAD_DIR}/*"):
    print(f"{os.path.basename(file)}: {os.path.getsize(file) / (1024**3):.2f} GB")

disk = shutil.disk_usage("/content")
print(f"\nFree disk space: {disk.free / (1024**3):.2f} GB")

Downloaded files:
human-screaming-detection-dataset.zip: 4.17 GB

Free disk space: 83.04 GB


In [ ]:
import zipfile
import glob
import os

zip_files = glob.glob("/content/companio_data/*.zip")

if not zip_files:
    print("ZIP file not found.")
else:
    zip_path = zip_files[0]

    with zipfile.ZipFile(zip_path, "r") as z:
        print("Archive integrity:", z.testzip() or "OK")
        print("\nFirst 30 archive entries:")
        for name in z.namelist()[:30]:
            print(name)

        print(f"\nTotal archive entries: {len(z.namelist())}")

Archive integrity: OK

First 30 archive entries:
NotScreaming/--PJHxphWEs_out.wav
NotScreaming/-28U1_qW0sU_out.wav
NotScreaming/-4xJv59_zcA_out.wav
NotScreaming/-5GhUbDLYkQ_out.wav
NotScreaming/-5Jlimvsuwo_out.wav
NotScreaming/-6RN8skIHU4_out.wav
NotScreaming/-6x2PtSRfJU_out.wav
NotScreaming/-75IwDlkzDQ_out.wav
NotScreaming/-7wUQP6G5EQ_out.wav
NotScreaming/-8Ezw9946g8_out.wav
NotScreaming/-ASYwidRD7M_out.wav
NotScreaming/-CgdUiJaNdU_out.wav
NotScreaming/-Dj2PfPmynQ_out.wav
NotScreaming/-Dtir74TiUM_out.wav
NotScreaming/-E013cWxoMo_out.wav
NotScreaming/-ECK_BisOLM_out.wav
NotScreaming/-FYyJ2sAWk4_out.wav
NotScreaming/-G_hnfp4a0M_out.wav
NotScreaming/-HN5kqCKpEA_out.wav
NotScreaming/-HnNrGtKXp4_out.wav
NotScreaming/-ItCGHv_9Gs_out.wav
NotScreaming/-J0H5ah1G7A_out.wav
NotScreaming/-JKLmqDk9p8_out.wav
NotScreaming/-KiAK66YwW8_out.wav
NotScreaming/-O842QWtymw_out.wav
NotScreaming/-QivqOnaPIc_out.wav
NotScreaming/-T-ZI1MZ44g_out.wav
NotScreaming/-Tea8DpWZQM_out.wav
NotScreaming/-WFsDr9xBVo_ou

In [ ]:
import zipfile
from collections import Counter

zip_path = "/content/companio_data/human-screaming-detection-dataset.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files = [
        name for name in z.namelist()
        if name.lower().endswith(".wav")
        and not name.endswith("/")
    ]

    class_counts = Counter(
        name.split("/")[0]
        for name in files
        if "/" in name
    )

print("Total WAV files:", len(files))
print("\nFiles per folder:")

for folder, count in class_counts.items():
    print(f"{folder}: {count}")

print("\nExample files from each folder:")
for folder in class_counts:
    examples = [name for name in files if name.startswith(folder + "/")]
    print(f"\n{folder}:")
    for name in examples[:5]:
        print(" ", name)

Total WAV files: 3493

Files per folder:
NotScreaming: 2631
Screaming: 862

Example files from each folder:

NotScreaming:
  NotScreaming/--PJHxphWEs_out.wav
  NotScreaming/-28U1_qW0sU_out.wav
  NotScreaming/-4xJv59_zcA_out.wav
  NotScreaming/-5GhUbDLYkQ_out.wav
  NotScreaming/-5Jlimvsuwo_out.wav

Screaming:
  Screaming/---1_cCGK4M_out.wav
  Screaming/-20uudT97E0_out.wav
  Screaming/-2yygHLdpXc_out.wav
  Screaming/-3bGlOhRkAo_out.wav
  Screaming/-4pUrlMafww_out.wav


In [ ]:
import os
import zipfile

zip_path = "/content/companio_data/human-screaming-detection-dataset.zip"
extract_dir = "/content/companio_data/audio"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

print("Extraction completed!")
print("Dataset location:", extract_dir)
print("Screaming folder:", os.path.join(extract_dir, "Screaming"))
print("NotScreaming folder:", os.path.join(extract_dir, "NotScreaming"))

print("\nScreaming clips:",
      len(os.listdir(os.path.join(extract_dir, "Screaming"))))

print("NotScreaming clips:",
      len(os.listdir(os.path.join(extract_dir, "NotScreaming"))))

Extraction completed!
Dataset location: /content/companio_data/audio
Screaming folder: /content/companio_data/audio/Screaming
NotScreaming folder: /content/companio_data/audio/NotScreaming

Screaming clips: 862
NotScreaming clips: 2631


In [ ]:
import os
import soundfile as sf

extract_dir = "/content/companio_data/audio"

for label in ["Screaming", "NotScreaming"]:
    folder = os.path.join(extract_dir, label)

    audio_files = [
        f for f in os.listdir(folder)
        if f.lower().endswith(".wav")
    ]

    print(f"\n{label}: {len(audio_files)} WAV files")

    for filename in audio_files[:3]:
        path = os.path.join(folder, filename)

        try:
            info = sf.info(path)
            print(
                f"{filename}: "
                f"{info.samplerate} Hz, "
                f"{info.channels} channel(s), "
                f"{info.duration:.2f} seconds"
            )
        except Exception as e:
            print(f"Could not inspect {filename}: {e}")


Screaming: 862 WAV files
60I8VwOkzcg_out.wav: 44100 Hz, 2 channel(s), 10.00 seconds
3UGVBYBbzz8_out.wav: 44100 Hz, 2 channel(s), 10.00 seconds
ywTqqfXcvmk_out.wav: 44100 Hz, 1 channel(s), 10.00 seconds

NotScreaming: 2631 WAV files
pfPWhjv_Wb0_out.wav: 44100 Hz, 2 channel(s), 10.00 seconds
qiPKWD2Q3kE_out.wav: 44100 Hz, 2 channel(s), 10.00 seconds
TcoRmHHHNgI_out.wav: 44100 Hz, 2 channel(s), 10.00 seconds


In [ ]:
%pip install -q soundfile

In [ ]:
%pip install -q tensorflow tensorflow-hub librosa soundfile scikit-learn pandas numpy matplotlib seaborn


In [ ]:
import os
import numpy as np
import soundfile as sf
import tensorflow as tf
import tensorflow_hub as hub
import sklearn

DATASET_DIR = "/content/companio_data/audio"

print("TensorFlow:", tf.__version__)
print("Scikit-learn:", sklearn.__version__)

for label in ["Screaming", "NotScreaming"]:
    folder = os.path.join(DATASET_DIR, label)

    files = [
        f for f in os.listdir(folder)
        if f.lower().endswith(".wav")
    ]

    print(f"\n{label}: {len(files)} audio files")

    # Check up to 20 clips for basic audio metadata.
    sample_rates = set()
    channel_counts = set()
    durations = []
    errors = []

    for filename in files[:20]:
        path = os.path.join(folder, filename)

        try:
            info = sf.info(path)
            sample_rates.add(info.samplerate)
            channel_counts.add(info.channels)
            durations.append(info.duration)
        except Exception as e:
            errors.append((filename, str(e)))

    print("Sample rates:", sample_rates)
    print("Channel counts:", channel_counts)

    if durations:
        print("Duration range:", round(min(durations), 2),
              "to", round(max(durations), 2), "seconds")

    print("Inspection errors:", len(errors))

/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


TensorFlow: 2.21.0
Scikit-learn: 1.6.1

Screaming: 862 audio files
Sample rates: {44100}
Channel counts: {1, 2}
Duration range: 6.5 to 10.0 seconds
Inspection errors: 0

NotScreaming: 2631 audio files
Sample rates: {44100}
Channel counts: {1, 2}
Duration range: 9.39 to 10.0 seconds
Inspection errors: 0


In [ ]:
import os
import numpy as np
import soundfile as sf
import librosa
import tensorflow as tf
import tensorflow_hub as hub

# Dataset location
DATASET_DIR = "/content/companio_data/audio"

# Load pretrained YAMNet
yamnet = hub.load(
    "https://tfhub.dev/google/yamnet/1"
)

print("YAMNet loaded successfully!")

# Select one screaming clip
screaming_folder = os.path.join(DATASET_DIR, "Screaming")
filename = next(
    f for f in os.listdir(screaming_folder)
    if f.lower().endswith(".wav")
)

audio_path = os.path.join(screaming_folder, filename)

# Read the audio file
waveform, sample_rate = sf.read(audio_path, dtype="float32")

# Convert stereo to mono
if waveform.ndim == 2:
    waveform = waveform.mean(axis=1)

# Resample from 44.1 kHz to 16 kHz
if sample_rate != 16000:
    waveform = librosa.resample(
        waveform,
        orig_sr=sample_rate,
        target_sr=16000
    )

waveform = waveform.astype(np.float32)

# Run YAMNet
scores, embeddings, spectrogram = yamnet(
    tf.convert_to_tensor(waveform)
)

print("\nTest clip:", filename)
print("Waveform shape:", waveform.shape)
print("Waveform sample rate: 16000 Hz")
print("YAMNet scores shape:", scores.shape)
print("Embeddings shape:", embeddings.shape)
print("Spectrogram shape:", spectrogram.shape)

YAMNet loaded successfully!

Test clip: 60I8VwOkzcg_out.wav
Waveform shape: (160000,)
Waveform sample rate: 16000 Hz
YAMNet scores shape: (20, 521)
Embeddings shape: (20, 1024)
Spectrogram shape: (1008, 64)


In [ ]:
import os
import pandas as pd

from sklearn.model_selection import train_test_split

DATASET_DIR = "/content/companio_data/audio"

records = []

label_mapping = {
    "NotScreaming": 0,
    "Screaming": 1
}

for folder, label in label_mapping.items():
    folder_path = os.path.join(DATASET_DIR, folder)

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".wav"):
            records.append({
                "filepath": os.path.join(folder_path, filename),
                "filename": filename,
                "label": label,
                "class_name": folder
            })

df = pd.DataFrame(records)

print("Total clips:", len(df))
print("\nClass distribution:")
print(df["class_name"].value_counts())

# First split: 70% training, 30% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

# Second split: divide temporary set equally
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("\nSplit sizes:")
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nClass distribution by split:")

for name, split_df in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}:")
    print(split_df["class_name"].value_counts())

Total clips: 3493

Class distribution:
class_name
NotScreaming    2631
Screaming        862
Name: count, dtype: int64

Split sizes:
Training: 2445
Validation: 524
Test: 524

Class distribution by split:

Training:
class_name
NotScreaming    1842
Screaming        603
Name: count, dtype: int64

Validation:
class_name
NotScreaming    394
Screaming       130
Name: count, dtype: int64

Test:
class_name
NotScreaming    395
Screaming       129
Name: count, dtype: int64


In [ ]:
import os
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import tensorflow as tf
import tensorflow_hub as hub
from tqdm.auto import tqdm

# Dataset location
DATASET_DIR = "/content/companio_data/audio"

# Load YAMNet if it is not already loaded
yamnet = hub.load("https://tfhub.dev/google/yamnet/1")

def extract_embedding(audio_path):
    """
    Load an audio clip, convert it to mono,
    resample to 16 kHz, and extract its mean YAMNet embedding.
    """
    waveform, sample_rate = sf.read(
        audio_path,
        dtype="float32"
    )

    # Convert stereo to mono
    if waveform.ndim == 2:
        waveform = waveform.mean(axis=1)

    # Resample to 16 kHz
    if sample_rate != 16000:
        waveform = librosa.resample(
            waveform,
            orig_sr=sample_rate,
            target_sr=16000
        )

    waveform = np.asarray(waveform, dtype=np.float32)

    # YAMNet returns scores, embeddings and spectrogram
    scores, embeddings, spectrogram = yamnet(
        tf.convert_to_tensor(waveform)
    )

    # Average frame embeddings into one vector per clip
    clip_embedding = tf.reduce_mean(
        embeddings,
        axis=0
    ).numpy()

    return clip_embedding


def extract_split_embeddings(split_df, split_name):
    """
    Extract embeddings for a given dataset split.
    """
    X = []
    y = []
    failed_files = []

    for row in tqdm(
        split_df.itertuples(index=False),
        total=len(split_df),
        desc=f"Processing {split_name}"
    ):
        try:
            embedding = extract_embedding(row.filepath)

            if not np.isfinite(embedding).all():
                raise ValueError("Embedding contains NaN or infinity")

            X.append(embedding)
            y.append(row.label)

        except Exception as e:
            failed_files.append({
                "filepath": row.filepath,
                "error": str(e)
            })

    if failed_files:
        print(
            f"\nWARNING: {len(failed_files)} files failed "
            f"during {split_name} extraction."
        )

    if not X:
        raise RuntimeError(
            f"No embeddings were extracted for {split_name}."
        )

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.int32),
        failed_files
    )


# Extract features for all three splits
X_train, y_train, failed_train = extract_split_embeddings(
    train_df, "training"
)

X_val, y_val, failed_val = extract_split_embeddings(
    val_df, "validation"
)

X_test, y_test, failed_test = extract_split_embeddings(
    test_df, "test"
)

print("\nEmbedding extraction complete!")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nFailed files:")
print("Training:", len(failed_train))
print("Validation:", len(failed_val))
print("Test:", len(failed_test))

Processing training:   0%|          | 0/2445 [00:00<?, ?it/s]

Processing validation:   0%|          | 0/524 [00:00<?, ?it/s]

Processing test:   0%|          | 0/524 [00:00<?, ?it/s]


Embedding extraction complete!
X_train: (2445, 1024)
y_train: (2445,)
X_val: (524, 1024)
y_val: (524,)
X_test: (524, 1024)
y_test: (524,)

Failed files:
Training: 0
Validation: 0
Test: 0


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
required_variables = [
    "val_df",
    "test_df",
    "y_val",
    "y_test",
    "yamnet"
]

for name in required_variables:
    print(f"{name}: {'Available' if name in globals() else 'Missing'}")

val_df: Missing
test_df: Missing
y_val: Missing
y_test: Missing
yamnet: Missing


In [3]:
import os

feature_dir = "/content/drive/MyDrive/Companio/features"

if os.path.isdir(feature_dir):
    print("Files found in Google Drive:")
    for filename in os.listdir(feature_dir):
        print("-", filename)
else:
    print("Companio/features folder not found.")

Companio/features folder not found.


In [1]:
import os
from google.colab import drive

drive.mount("/content/drive")

print("\nMy Drive accessible:", os.path.exists("/content/drive/MyDrive"))
print("\nTop-level files and folders in My Drive:")

for item in os.listdir("/content/drive/MyDrive")[:50]:
    print(item)

Mounted at /content/drive

My Drive accessible: True

Top-level files and folders in My Drive:
Colab Notebooks
Companio


In [2]:
import os

companio_dir = "/content/drive/MyDrive/Companio"

for current_dir, dirs, files in os.walk(companio_dir):
    level = current_dir.replace(companio_dir, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(current_dir)}/")

    for filename in files:
        print(f"{indent}  {filename}")

Companio/
  features/
    yamnet_embeddings.npz


In [3]:
import os
import numpy as np

FEATURE_PATH = (
    "/content/drive/MyDrive/Companio/features/"
    "yamnet_embeddings.npz"
)

data = np.load(FEATURE_PATH)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

X_test = data["X_test"]
y_test = data["y_test"]

print("Embeddings loaded successfully!")

print("\nTraining:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Embeddings loaded successfully!

Training: (2445, 1024) (2445,)
Validation: (524, 1024) (524,)
Test: (524, 1024) (524,)


In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Create the baseline trainable classifier
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )
)

# Train on the training embeddings only
model.fit(X_train, y_train)

print("Classifier training completed!")
print("Training samples:", len(X_train))
print("Features per sample:", X_train.shape[1])

Classifier training completed!
Training samples: 2445
Features per sample: 1024


In [5]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

# Predict probabilities on validation data
val_probabilities = model.predict_proba(X_val)[:, 1]

# Use 0.5 as the initial threshold
val_predictions = (val_probabilities >= 0.5).astype(int)

print("COMPANIO — VALIDATION RESULTS")
print("=" * 45)

print("Accuracy:", round(accuracy_score(y_val, val_predictions), 4))
print("Precision:", round(precision_score(y_val, val_predictions, zero_division=0), 4))
print("Recall:", round(recall_score(y_val, val_predictions, zero_division=0), 4))
print("F1-score:", round(f1_score(y_val, val_predictions, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_val, val_probabilities), 4))
print("PR-AUC:", round(average_precision_score(y_val, val_probabilities), 4))

print("\nClassification report:")
print(classification_report(
    y_val,
    val_predictions,
    target_names=["NotScreaming", "Screaming"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_val, val_predictions))

COMPANIO — VALIDATION RESULTS
Accuracy: 0.9065
Precision: 0.8045
Recall: 0.8231
F1-score: 0.8137
ROC-AUC: 0.9495
PR-AUC: 0.9002

Classification report:
              precision    recall  f1-score   support

NotScreaming       0.94      0.93      0.94       394
   Screaming       0.80      0.82      0.81       130

    accuracy                           0.91       524
   macro avg       0.87      0.88      0.88       524
weighted avg       0.91      0.91      0.91       524

Confusion matrix:
[[368  26]
 [ 23 107]]


In [6]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

thresholds = np.arange(0.10, 0.91, 0.05)

results = []

for threshold in thresholds:
    predictions = (
        val_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        predictions,
        labels=[0, 1]
    ).ravel()

    results.append({
        "Threshold": round(float(threshold), 2),
        "Precision": round(
            precision_score(
                y_val,
                predictions,
                zero_division=0
            ), 4
        ),
        "Recall": round(
            recall_score(
                y_val,
                predictions,
                zero_division=0
            ), 4
        ),
        "F1": round(
            f1_score(
                y_val,
                predictions,
                zero_division=0
            ), 4
        ),
        "False Positives": int(fp),
        "Missed Screams": int(fn)
    })

threshold_results = pd.DataFrame(results)

print(threshold_results.to_string(index=False))

 Threshold  Precision  Recall     F1  False Positives  Missed Screams
      0.10     0.6573  0.9000 0.7597               61              13
      0.15     0.6746  0.8769 0.7625               55              16
      0.20     0.7089  0.8615 0.7778               46              18
      0.25     0.7417  0.8615 0.7972               39              18
      0.30     0.7500  0.8538 0.7986               37              19
      0.35     0.7586  0.8462 0.8000               35              20
      0.40     0.7842  0.8385 0.8104               30              21
      0.45     0.7868  0.8231 0.8045               29              23
      0.50     0.8045  0.8231 0.8137               26              23
      0.55     0.8231  0.8231 0.8231               23              23
      0.60     0.8217  0.8154 0.8185               23              24
      0.65     0.8443  0.7923 0.8175               19              27
      0.70     0.8487  0.7769 0.8112               18              29
      0.75     0.855

## Operating Threshold Selection — Validation Set

### Selection Rule

The MEDIUM operating threshold is selected using the validation set only.

**Rule:** Select the highest evaluated threshold that achieves a validation recall of at least 85%.

This prioritizes detecting distress while reducing unnecessary alerts as much as possible under the chosen recall requirement.

### Selected MEDIUM Threshold

Based on the validation threshold analysis:

- **Selected threshold:** 0.30
- **Validation precision:** 75.00%
- **Validation recall:** 85.38%
- **False positives:** 37
- **Missed screams:** 19

At threshold 0.35, validation recall falls to 84.62%, below the 85% requirement. Therefore, 0.30 is the highest evaluated threshold satisfying the selection rule.

### Important Evaluation Note

Threshold 0.20 was used for an earlier test-set evaluation during development. Because the test results have already been inspected, the test set must not be used to select or optimize the replacement threshold.

The threshold of 0.30 is selected from validation results only. The previously reported test metrics at 0.20 must not be presented as test performance for the 0.30 configuration.

This threshold is an experimental operating point, not a validated setting for automatically triggering emergency SOS alerts.

## Amendment 2: Three Confidence Bands for Distress Detection

### Objective

Companio will classify the model's predicted scream probability into three experimental confidence bands: LOW, MEDIUM, and HIGH.

These bands are intended to support different levels of user notification. A higher model score indicates stronger evidence of a scream according to the trained classifier, but it does not establish that an emergency is occurring.

### 1. LOW Confidence

**Rule:** Predicted scream probability < 0.30

Interpretation:
- The model's score is below the selected MEDIUM threshold.
- No scream alert is generated by this experimental classification rule.
- This does not prove that the user is safe or that no distress is occurring.

### 2. MEDIUM Confidence

**Rule:** Predicted scream probability >= 0.30 and < 0.90

Interpretation:
- The score has crossed the MEDIUM threshold.
- The app may notify the user and offer an immediate way to trigger SOS.
- This band is intended for further evaluation because false positives may occur.

### 3. HIGH Confidence

**Rule:** Predicted scream probability >= 0.90

Interpretation:
- The score has crossed the HIGH threshold.
- The validation threshold sweep identified 0.90 as the evaluated threshold achieving at least 90% precision.
- This operating point has a higher precision target but may miss more actual screams.

### Threshold Selection Rules

- **MEDIUM threshold:** Highest evaluated threshold achieving validation recall >= 85%, which is 0.30.
- **HIGH threshold:** Lowest evaluated threshold achieving validation precision >= 90%, which is 0.90.

These thresholds are selected from validation results, not optimized using the test set.

### Safety Limitation

These confidence bands are experimental classifications, not proof of an emergency.

The current model has not been validated for reliable automatic emergency detection in real-world environments. A HIGH score must not be treated as sufficient evidence to automatically contact police or emergency services.

Before deployment, Companio requires testing in realistic environments, evaluation of false alarms and missed screams, and a carefully designed SOS confirmation and escalation mechanism.

In [14]:
# Companio: Three experimental confidence bands

MEDIUM_THRESHOLD = 0.30
HIGH_THRESHOLD = 0.90


def classify_distress_confidence(probability):
    """
    Classify the model's predicted scream probability.

    These bands are experimental and are not proof of an emergency.
    """
    if probability < MEDIUM_THRESHOLD:
        return "LOW"
    elif probability < HIGH_THRESHOLD:
        return "MEDIUM"
    else:
        return "HIGH"


# Demonstrate the classification rules
test_probabilities = [0.10, 0.29, 0.30, 0.55, 0.89, 0.90, 0.97]

for probability in test_probabilities:
    band = classify_distress_confidence(probability)
    print(f"Probability: {probability:.2f} -> {band}")

Probability: 0.10 -> LOW
Probability: 0.29 -> LOW
Probability: 0.30 -> MEDIUM
Probability: 0.55 -> MEDIUM
Probability: 0.89 -> MEDIUM
Probability: 0.90 -> HIGH
Probability: 0.97 -> HIGH


## Amendment 3: Confidence Intervals and Evaluation Reliability

### Objective

Point estimates such as precision, recall, and false-positive rate describe the observed performance of the classifier on a particular dataset. However, these estimates have sampling uncertainty.

This section reports 95% confidence intervals (CIs) alongside the evaluation metrics to communicate that uncertainty.

### Previously Observed Test Results

The earlier test-set evaluation used a threshold of 0.20 and produced the following results:

- **Test accuracy:** 88.55%
- **Test precision:** 69.94%
- **Test recall:** 93.80%
- **Test F1-score:** 80.13%
- **Test ROC-AUC:** 96.93%
- **False positives:** 52 out of 395 non-scream clips
- **Missed screams:** 8 out of 129 scream clips

The estimated false-positive rate was 13.16%.

### Interpretation

The observed recall indicates that the classifier detected a large proportion of scream clips in this test set. However, the false-positive rate indicates that some non-scream clips were incorrectly classified as screams.

In a real-world safety application, false positives may cause unnecessary alerts, while false negatives may result in missed distress events.

### Important Evaluation Limitation

The test set was inspected during development using a threshold of 0.20. Therefore, these results should be reported as an exploratory test evaluation, not as an untouched final evaluation.

The test results at threshold 0.20 must not be presented as the performance of the newly selected threshold of 0.30.

Confidence intervals describe sampling uncertainty on the evaluated dataset. They do not establish real-world reliability or guarantee future performance.

In [15]:
import math
import pandas as pd


def wilson_confidence_interval(successes, total, z=1.96):
    """
    Calculate a 95% Wilson confidence interval for a proportion.
    """
    if total <= 0:
        return float("nan"), float("nan")

    p = successes / total
    denominator = 1 + (z**2 / total)

    center = (
        p + z**2 / (2 * total)
    ) / denominator

    margin = (
        z
        * math.sqrt(
            (p * (1 - p) / total)
            + (z**2 / (4 * total**2))
        )
        / denominator
    )

    return max(0, center - margin), min(1, center + margin)


# Previously observed test-set confusion-matrix counts
# TN = 343, FP = 52, FN = 8, TP = 121

TN = 343
FP = 52
FN = 8
TP = 121

metrics = {
    "Precision": (TP, TP + FP),
    "Recall": (TP, TP + FN),
    "False Positive Rate": (FP, FP + TN)
}

ci_results = []

for metric_name, (successes, total) in metrics.items():
    estimate = successes / total

    lower, upper = wilson_confidence_interval(
        successes,
        total
    )

    ci_results.append({
        "Metric": metric_name,
        "Estimate (%)": round(estimate * 100, 2),
        "95% CI Lower (%)": round(lower * 100, 2),
        "95% CI Upper (%)": round(upper * 100, 2),
        "Numerator": successes,
        "Denominator": total
    })

ci_table = pd.DataFrame(ci_results)

print(ci_table.to_string(index=False))

             Metric  Estimate (%)  95% CI Lower (%)  95% CI Upper (%)  Numerator  Denominator
          Precision         69.94             62.74             76.28        121          173
             Recall         93.80             88.24             96.82        121          129
False Positive Rate         13.16             10.18             16.86         52          395


### Confidence Interval Interpretation

The 95% Wilson confidence intervals quantify uncertainty around the observed precision, recall, and false-positive rate.

Test recall was 93.80% (95% CI: 88.24%–96.82%), while precision was 69.94% (95% CI: 62.74%–76.28%). The false-positive rate was 13.16% (95% CI: 10.18%–16.86%).

These results indicate promising scream-detection performance on the evaluated dataset, alongside a meaningful false-alarm risk.

The test set was inspected during development at a threshold of 0.20. These results are therefore exploratory and must not be represented as an untouched final evaluation or as performance at the selected threshold of 0.30.

Confidence intervals do not establish real-world reliability. Further evaluation using independent recordings and realistic environmental conditions is required before deployment.

## Amendment 4: Dataset Limitations and Real-World Safety Risks

### 1. Dataset Class Imbalance

The dataset contains 3,493 audio clips:

- Screaming: 862 clips
- NotScreaming: 2,631 clips

Approximately 24.7% of the dataset contains screaming, while 75.3% contains non-screaming audio.

The proportion of scream clips in this dataset may differ substantially from the proportion of emergency events encountered during everyday use.

Therefore, precision measured on this dataset may not represent the precision achieved in real-world environments.

### 2. Potential Pretraining Overlap

The scream classifier uses embeddings extracted from YAMNet, a pretrained audio model.

The possibility of overlap between the evaluation dataset and the pretrained model's training data should be investigated. Some audio filenames resemble YouTube video identifiers, but this alone does not establish that the recordings overlap with YAMNet's training data.

Until the dataset's provenance and any potential overlap are verified, this remains a limitation to investigate rather than a confirmed source of data leakage.

### 3. False Alarms in Real-World Environments

The earlier test evaluation at threshold 0.20 produced 52 false positives among 395 non-scream clips.

This corresponds to a false-positive rate of approximately 13.16%.

A false positive occurs when the model classifies a non-scream clip as a scream.

Potential causes include loud conversations, shouting, background television, music, and other sounds that resemble screaming. These examples are possible failure modes and require experimental verification.

### 4. Illustrative False Alarms per Hour

If the measured 13.16% false-positive rate remained constant across independent, non-scream audio windows of 10 seconds each, the estimated number of false-positive windows would be approximately 47 per hour.

This is a mathematical illustration, not a measured real-world false-alarm rate.

Actual performance will depend on the audio environment, window duration, overlap between windows, event aggregation, and the deployed detection threshold.

### 5. Safety Implications

Companio must evaluate both missed distress events and unnecessary alerts.

The current classifier is an experimental prototype. It has not yet been validated for continuous real-world monitoring or automatic emergency escalation.

Before deployment, further testing should include diverse speakers, environmental noise, everyday activities, longer recordings, and independent data sources.

The application should also provide a reliable manual SOS mechanism that remains available when scream detection fails.

In [16]:
# Companio: Illustrative false-positive estimate per hour

FALSE_POSITIVES = 52
NON_SCREAM_CLIPS = 395

CLIP_DURATION_SECONDS = 10
SECONDS_PER_HOUR = 3600

false_positive_rate = (
    FALSE_POSITIVES / NON_SCREAM_CLIPS
)

windows_per_hour = (
    SECONDS_PER_HOUR / CLIP_DURATION_SECONDS
)

estimated_false_positive_windows_per_hour = (
    false_positive_rate * windows_per_hour
)

print(f"False-positive rate: {false_positive_rate:.2%}")

print(
    "Illustrative false-positive windows per hour: "
    f"{estimated_false_positive_windows_per_hour:.1f}"
)

print(
    "\nIMPORTANT: This is a mathematical illustration, "
    "not a measured real-world alert rate."
)

False-positive rate: 13.16%
Illustrative false-positive windows per hour: 47.4

IMPORTANT: This is a mathematical illustration, not a measured real-world alert rate.


In [7]:
import os
import numpy as np
import pandas as pd

FEATURE_DIR = "/content/drive/MyDrive/Companio/features"
os.makedirs(FEATURE_DIR, exist_ok=True)

threshold_results.to_csv(
    os.path.join(FEATURE_DIR, "threshold_analysis.csv"),
    index=False
)

print("Threshold analysis saved successfully!")
print("Location:", FEATURE_DIR)

Threshold analysis saved successfully!
Location: /content/drive/MyDrive/Companio/features


In [8]:
# Candidate threshold selected using validation results
CANDIDATE_THRESHOLD = 0.20

print("Candidate threshold:", CANDIDATE_THRESHOLD)
print("Validation recall: 86.15%")
print("Validation missed screams: 18 out of 130")
print("Validation false positives: 46 out of 394")

Candidate threshold: 0.2
Validation recall: 86.15%
Validation missed screams: 18 out of 130
Validation false positives: 46 out of 394


In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Use the candidate threshold selected from validation
CANDIDATE_THRESHOLD = 0.20

# Predict probabilities for the unseen test set
y_test_prob = model.predict_proba(X_test)[:, 1]

# Convert probabilities into predictions
y_test_pred = (y_test_prob >= CANDIDATE_THRESHOLD).astype(int)

# Evaluate the model
print("COMPANIO — TEST SET EVALUATION")
print("=" * 40)

print(f"Candidate threshold: {CANDIDATE_THRESHOLD}")
print(f"Test accuracy:  {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Test precision: {precision_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"Test recall:    {recall_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"Test F1-score:  {f1_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"Test ROC-AUC:   {roc_auc_score(y_test, y_test_prob):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["NotScreaming", "Screaming"],
    zero_division=0
))

COMPANIO — TEST SET EVALUATION
Candidate threshold: 0.2
Test accuracy:  0.8855
Test precision: 0.6994
Test recall:    0.9380
Test F1-score:  0.8013
Test ROC-AUC:   0.9693

Confusion Matrix:
[[343  52]
 [  8 121]]

Classification Report:
              precision    recall  f1-score   support

NotScreaming       0.98      0.87      0.92       395
   Screaming       0.70      0.94      0.80       129

    accuracy                           0.89       524
   macro avg       0.84      0.90      0.86       524
weighted avg       0.91      0.89      0.89       524



In [17]:
import os
import json

MODEL_DIR = "/content/drive/MyDrive/Companio/models"
os.makedirs(MODEL_DIR, exist_ok=True)

metadata_path = os.path.join(
    MODEL_DIR,
    "model_metadata.json"
)

metadata = {
    "model_name": "Companio Scream Detection",
    "feature_extractor": "YAMNet",
    "embedding_configuration": {
        "method": "mean pooling",
        "embedding_dimensions": 1024
    },
    "classifier": "Logistic Regression",

    "operating_thresholds": {
        "low": {
            "condition": "probability < 0.30"
        },
        "medium": {
            "condition": "0.30 <= probability < 0.90",
            "threshold": 0.30,
            "selection_rule": (
                "Highest evaluated validation threshold "
                "with recall >= 0.85"
            ),
            "validation_precision": 0.7500,
            "validation_recall": 0.8538,
            "validation_false_positives": 37,
            "validation_missed_screams": 19
        },
        "high": {
            "condition": "probability >= 0.90",
            "threshold": 0.90,
            "selection_rule": (
                "Lowest evaluated validation threshold "
                "with precision >= 0.90"
            ),
            "validation_precision": 0.9100,
            "validation_recall": 0.7000
        }
    },

    "exploratory_test_evaluation": {
        "threshold": 0.20,
        "test_accuracy": 0.8855,
        "test_precision": 0.6994,
        "test_recall": 0.9380,
        "test_f1_score": 0.8013,
        "test_roc_auc": 0.9693,
        "test_samples": 524,
        "test_true_positives": 121,
        "test_true_negatives": 343,
        "test_false_positives": 52,
        "test_false_negatives": 8,

        "confidence_intervals_95": {
            "precision": [0.6274, 0.7628],
            "recall": [0.8824, 0.9682],
            "false_positive_rate": [0.1018, 0.1686]
        },

        "false_positive_rate": 0.1316,
        "test_set_inspected_during_development": True
    },

    "evaluation_limitations": [
        "Test results were inspected during development.",
        "Test metrics at threshold 0.20 do not represent "
        "performance at threshold 0.30 or 0.90.",
        "Potential overlap between YAMNet pretraining data "
        "and the evaluation dataset has not been verified.",
        "Real-world false-alarm rates have not been measured.",
        "Confidence intervals assume reasonably independent "
        "evaluation observations."
    ],

    "deployment_status": (
        "Research prototype; not validated for emergency use"
    ),

    "automatic_sos_approved": False
}

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print("Metadata updated:", metadata_path)
print("Classifier file was not modified.")

Metadata updated: /content/drive/MyDrive/Companio/models/model_metadata.json
Classifier file was not modified.


In [18]:
import json

with open(metadata_path, "r") as f:
    saved_metadata = json.load(f)

print("MEDIUM threshold:",
      saved_metadata["operating_thresholds"]["medium"]["threshold"])

print("HIGH threshold:",
      saved_metadata["operating_thresholds"]["high"]["threshold"])

print("Previous test threshold:",
      saved_metadata["exploratory_test_evaluation"]["threshold"])

print("Automatic SOS approved:",
      saved_metadata["automatic_sos_approved"])

print("Deployment status:",
      saved_metadata["deployment_status"])

MEDIUM threshold: 0.3
HIGH threshold: 0.9
Previous test threshold: 0.2
Automatic SOS approved: False
Deployment status: Research prototype; not validated for emergency use


In [19]:
import os
import joblib
import json
import numpy as np

MODEL_DIR = "/content/drive/MyDrive/Companio/models"

model_path = os.path.join(
    MODEL_DIR,
    "companio_scream_classifier.joblib"
)

metadata_path = os.path.join(
    MODEL_DIR,
    "model_metadata.json"
)

# Verify that both files exist
assert os.path.exists(model_path), "Saved classifier not found."
assert os.path.exists(metadata_path), "Metadata file not found."

# Load the saved classifier without retraining
loaded_model = joblib.load(model_path)

# Load the metadata
with open(metadata_path, "r") as f:
    saved_metadata = json.load(f)

# Verify the classifier's expected input dimensions
assert loaded_model.n_features_in_ == 1024, (
    "Unexpected feature dimensions."
)

print("Saved classifier loaded successfully.")
print("Expected input dimensions:", loaded_model.n_features_in_)
print("Classifier:", saved_metadata["classifier"])
print("Feature extractor:", saved_metadata["feature_extractor"])
print("MEDIUM threshold:",
      saved_metadata["operating_thresholds"]["medium"]["threshold"])
print("HIGH threshold:",
      saved_metadata["operating_thresholds"]["high"]["threshold"])
print("Model verification passed.")

Saved classifier loaded successfully.
Expected input dimensions: 1024
Classifier: Logistic Regression
Feature extractor: YAMNet
MEDIUM threshold: 0.3
HIGH threshold: 0.9
Model verification passed.


In [20]:
# Amendment 7: Saved-model prediction verification

import numpy as np

# Confirm the existing test embeddings are available
assert "X_test" in globals(), (
    "X_test is not available. Do not recreate the split; "
    "we will handle this if necessary."
)

# Generate probabilities using the saved classifier
test_probabilities_check = loaded_model.predict_proba(
    X_test
)[:, 1]

# Verify prediction shape and validity
assert len(test_probabilities_check) == len(X_test)
assert np.all(np.isfinite(test_probabilities_check))
assert np.all(test_probabilities_check >= 0)
assert np.all(test_probabilities_check <= 1)

# Classify using the saved operating thresholds
test_bands = [
    classify_distress_confidence(float(p))
    for p in test_probabilities_check
]

print("Prediction verification passed.")
print("Number of test predictions:", len(test_probabilities_check))
print(
    "Probability range:",
    round(float(test_probabilities_check.min()), 4),
    "to",
    round(float(test_probabilities_check.max()), 4)
)

print("\nConfidence-band counts:")
for band in ["LOW", "MEDIUM", "HIGH"]:
    print(f"{band}: {test_bands.count(band)}")

print("\nNo model retraining or modification performed.")

Prediction verification passed.
Number of test predictions: 524
Probability range: 0.0 to 1.0

Confidence-band counts:
LOW: 361
MEDIUM: 60
HIGH: 103

No model retraining or modification performed.


In [21]:
# Amendment 8: Compare original and saved-model predictions

assert "model" in globals(), (
    "Original model is unavailable. "
    "Do not retrain it just for this check."
)

original_probabilities = model.predict_proba(X_test)[:, 1]

saved_probabilities = loaded_model.predict_proba(X_test)[:, 1]

np.testing.assert_allclose(
    original_probabilities,
    saved_probabilities,
    rtol=1e-7,
    atol=1e-9
)

print("Prediction consistency verified.")
print("Original and saved models produce matching probabilities.")
print("No retraining performed.")

Prediction consistency verified.
Original and saved models produce matching probabilities.
No retraining performed.


In [22]:
# Amendment 9: Verify current metadata thresholds

import json
import os

MODEL_DIR = "/content/drive/MyDrive/Companio/models"

metadata_path = os.path.join(
    MODEL_DIR,
    "model_metadata.json"
)

with open(metadata_path, "r") as f:
    metadata_check = json.load(f)

medium_threshold = metadata_check[
    "operating_thresholds"
]["medium"]["threshold"]

high_threshold = metadata_check[
    "operating_thresholds"
]["high"]["threshold"]

previous_test_threshold = metadata_check[
    "exploratory_test_evaluation"
]["threshold"]

print("CURRENT THRESHOLD VERIFICATION")
print("=" * 40)

print("MEDIUM threshold:", medium_threshold)
print("HIGH threshold:", high_threshold)
print("Previous exploratory test threshold:", previous_test_threshold)

assert medium_threshold == 0.30
assert high_threshold == 0.90
assert previous_test_threshold == 0.20

print("\nSUCCESS: Metadata thresholds are consistent.")
print("The saved classifier was not modified.")

CURRENT THRESHOLD VERIFICATION
MEDIUM threshold: 0.3
HIGH threshold: 0.9
Previous exploratory test threshold: 0.2

SUCCESS: Metadata thresholds are consistent.
The saved classifier was not modified.


## Final Results Summary — Companio Scream Detection

### 1. Model Architecture

- Audio feature extractor: YAMNet
- Embedding method: Mean pooling
- Embedding dimensions: 1,024
- Classifier: StandardScaler + Logistic Regression
- Class weighting: Balanced
- Training approach: Transfer learning using pretrained audio embeddings

### 2. Dataset

- Total audio clips: 3,493
- Screaming clips: 862
- Non-screaming clips: 2,631
- Training samples: 2,445
- Validation samples: 524
- Test samples: 524

### 3. Experimental Operating Thresholds

- LOW: Probability below 0.30
- MEDIUM: Probability from 0.30 to below 0.90
- HIGH: Probability of at least 0.90

The MEDIUM threshold was selected using a validation-recall requirement of at least 85%.

The HIGH threshold was selected using a validation-precision requirement of at least 90%.

### 4. Previously Observed Test Results

At the exploratory threshold of 0.20:

- Accuracy: 88.55%
- Precision: 69.94%
- Recall: 93.80%
- F1-score: 80.13%
- ROC-AUC: 96.93%
- False positives: 52
- Missed screams: 8

The 95% Wilson confidence intervals were:

- Precision: 62.74%–76.28%
- Recall: 88.24%–96.82%
- False-positive rate: 10.18%–16.86%

These results belong to the earlier evaluation at threshold 0.20. They do not establish performance at the selected MEDIUM threshold of 0.30 or the HIGH threshold of 0.90.

### 5. Artifact Verification

- Saved classifier exists and loads successfully.
- Expected input dimension: 1,024.
- Original and reloaded model predictions match.
- Metadata records the MEDIUM and HIGH thresholds.
- The classifier was not retrained during the verification amendments.

### 6. Limitations and Future Work

- The test set was inspected during development; an untouched final evaluation is still needed.
- Real-world false-alarm rates have not been measured.
- Potential overlap with YAMNet's pretraining data remains unverified.
- Independent recordings and realistic environmental conditions must be evaluated.
- Continuous monitoring, on-device inference, battery consumption, latency, and background-noise robustness remain to be tested.
- Confidence bands do not establish that an emergency is occurring.
- Automatic emergency escalation is not approved for this prototype.

### Conclusion

Companio has a working experimental scream-classification pipeline using pretrained audio embeddings and Logistic Regression.

The classifier, threshold configuration, and metadata have been saved and verified.

Further independent evaluation and safety testing are required before integrating scream detection into a real-world emergency-response workflow.

In [23]:
import os
import joblib
import numpy as np
import json

MODEL_DIR = "/content/drive/MyDrive/Companio/models"

# Load the saved classifier
model_path = os.path.join(
    MODEL_DIR,
    "companio_scream_classifier.joblib"
)

loaded_model = joblib.load(model_path)

# Load model metadata
metadata_path = os.path.join(
    MODEL_DIR,
    "model_metadata.json"
)

with open(metadata_path, "r") as f:
    metadata = json.load(f)

# Generate predictions using the original and loaded models
original_predictions = model.predict_proba(X_test)[:, 1]
loaded_predictions = loaded_model.predict_proba(X_test)[:, 1]

# Verify prediction consistency
predictions_match = np.allclose(
    original_predictions,
    loaded_predictions,
    rtol=1e-5,
    atol=1e-7
)

print("MODEL RELOAD VERIFICATION")
print("=" * 35)
print("Model file exists:", os.path.exists(model_path))
print("Metadata file exists:", os.path.exists(metadata_path))
print("Model type:", type(loaded_model).__name__)
print(
    "MEDIUM threshold:",
    metadata["operating_thresholds"]["medium"]["threshold"]
)
print(
    "HIGH threshold:",
    metadata["operating_thresholds"]["high"]["threshold"]
)
print(
    "Previous exploratory test threshold:",
    metadata["exploratory_test_evaluation"]["threshold"]
)
print("Predictions match:", predictions_match)

if predictions_match:
    print("\nSUCCESS: Saved model reloads correctly!")
else:
    print("\nWARNING: Predictions differ. Investigate before integration.")

MODEL RELOAD VERIFICATION
Model file exists: True
Metadata file exists: True
Model type: Pipeline
MEDIUM threshold: 0.3
HIGH threshold: 0.9
Previous exploratory test threshold: 0.2
Predictions match: True

SUCCESS: Saved model reloads correctly!


In [12]:
import tensorflow as tf
import tensorflow_hub as hub
import joblib
import numpy as np

print("TensorFlow:", tf.__version__)
print("TensorFlow Hub:", hub.__version__)
print("NumPy:", np.__version__)
print("Joblib:", joblib.__version__)

print("\nClassifier loaded:", "loaded_model" in globals())
print("YAMNet loaded:", "yamnet" in globals())

/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


TensorFlow: 2.21.0
TensorFlow Hub: 0.16.1
NumPy: 2.1.3
Joblib: 1.6.0

Classifier loaded: True
YAMNet loaded: False
